In [60]:
import numpy as np
import pandas as pd
import fastf1
from typing import Dict, List, Tuple


fastf1.Cache.enable_cache(".fastf1_cache")

In [61]:
session = fastf1.get_session(2024, "Monaco", "R")
session.load(telemetry=True)

core           INFO 	Loading data for Monaco Grand Prix - Race [v3.6.1]
req            INFO 	Using cached data for session_info
req            INFO 	Using cached data for driver_info
req            INFO 	Using cached data for session_status_data
req            INFO 	Using cached data for lap_count
req            INFO 	Using cached data for track_status_data
req            INFO 	Using cached data for _extended_timing_data
req            INFO 	Using cached data for timing_app_data
core           INFO 	Processing timing data...
req            INFO 	Using cached data for car_data
req            INFO 	Using cached data for position_data
req            INFO 	Using cached data for weather_data
req            INFO 	Using cached data for race_control_messages
core           INFO 	Finished loading data for 20 drivers: ['16', '81', '55', '4', '63', '1', '44', '22', '23', '10', '14', '3', '77', '18', '2', '24', '31', '11', '27', '20']


In [62]:
def telemetry_time_seconds(df: pd.DataFrame) -> pd.Series:
    if 'Time' in df.columns:
        ser = df['Time']
        if pd.api.types.is_timedelta64_dtype(ser.dtype):
            return ser.dt.total_seconds().astype(float).reset_index(drop=True)
        if pd.api.types.is_datetime64_any_dtype(ser.dtype):
            ref = ser.iloc[0]
            return (ser - ref).dt.total_seconds().astype(float).reset_index(drop=True)

    if isinstance(df.index, pd.DatetimeIndex):
        ref = df.index[0]
        return (df.index - ref).total_seconds().astype(float)
    if isinstance(df.index, pd.TimedeltaIndex):
        return df.index.total_seconds().astype(float)

    for col in df.columns:
        if 'time' in col.lower():
            ser = df[col]
            try:
                if pd.api.types.is_timedelta64_dtype(ser.dtype):
                    return ser.dt.total_seconds().astype(float).reset_index(drop=True)
                if pd.api.types.is_datetime64_any_dtype(ser.dtype):
                    ref = ser.iloc[0]
                    return (ser - ref).dt.total_seconds().astype(float).reset_index(drop=True)
                if pd.api.types.is_numeric_dtype(ser.dtype):
                    return ser.astype(float).reset_index(drop=True)
            except Exception:
                pass

    return None


def telemetry_absolute_time(lap) -> pd.Series:
    """
    Convert lap telemetry to absolute time in seconds from session start.
    """
    # Obtain telemetry DataFrame from lap object or dict-like row
    telem = None
    if hasattr(lap, 'get_telemetry'):
        try:
            telem = lap.get_telemetry()
        except Exception:
            telem = None
    elif isinstance(lap, dict) and 'Lap' in lap:
        try:
            telem = lap['Lap'].get_telemetry()
        except Exception:
            telem = None

    tsec = telemetry_time_seconds(telem) if telem is not None else None

    # Determine lap start value robustly. lap may be a Series (row) where 'Time' is a scalar
    lap_start_val = None
    if isinstance(lap, (pd.Series, dict)) and 'Time' in lap:
        lap_start_val = lap['Time']
    else:
        # fall back to the first index entry from telemetry (Timedelta or Timestamp)
        lap_start_val = telem.index[0] if telem is not None and len(telem.index) > 0 else 0

    # Convert lap_start_val to seconds (float)
    if isinstance(lap_start_val, pd.Timedelta):
        lap_start_sec = lap_start_val.total_seconds()
    elif isinstance(lap_start_val, (pd.Timestamp,)) or pd.api.types.is_datetime64_any_dtype(type(lap_start_val)):
        # If it's a Timestamp, we don't have a session-origin here — keep zero as baseline
        try:
            lap_start_sec = (lap_start_val - lap_start_val).total_seconds()  # zero-based fallback
        except Exception:
            lap_start_sec = 0.0
    else:
        try:
            lap_start_sec = float(lap_start_val)
        except Exception:
            lap_start_sec = 0.0

    absolute_time = tsec + lap_start_sec if tsec is not None else None
    return absolute_time


def concat_laps_telemetry(laps: pd.DataFrame) -> List[Tuple[int, pd.DataFrame, pd.Series]]:
    results = []
    for i, lap in laps.iterrows():
        try:
            telem = lap['Lap'].get_telemetry() if 'Lap' in lap else lap.get_telemetry()
        except Exception as e:
            print(f"Warning: couldn't get telemetry for lap index {i}: {e}")
            continue

        abs_time = telemetry_absolute_time(lap['Lap'] if 'Lap' in lap else lap)
        telem = telem.reset_index(drop=True)
        if len(abs_time) != len(telem):
            minlen = min(len(abs_time), len(telem))
            telem = telem.iloc[:minlen].reset_index(drop=True)
            abs_time = abs_time[:minlen]

        results.append((i, telem, abs_time))
    return results


In [63]:
INTERPOLATE_COLS = [
    "Speed", "RPM", "Throttle", "Brake",
    "Distance", "RelativeDistance",
    "X", "Y", "Z",
]

CATEGORICAL_COLS = [
    "nGear", "DRS", "Source", "Status",
    "DriverAhead", "lap_idx"
]

In [64]:
laps = session.laps
drivers = sorted(laps['Driver'].unique())
print(f"Found {len(drivers)} drivers in session: {drivers}")

Found 20 drivers in session: ['ALB', 'ALO', 'BOT', 'GAS', 'HAM', 'HUL', 'LEC', 'MAG', 'NOR', 'OCO', 'PER', 'PIA', 'RIC', 'RUS', 'SAI', 'SAR', 'STR', 'TSU', 'VER', 'ZHO']


In [65]:
all_lap_telem = []
driver_lap_map: Dict[str, List[Tuple[int, pd.DataFrame, pd.Series]]] = {}

In [66]:
# for d in drivers:
for d in ["LEC"]:
    d_laps = laps[laps['Driver'] == d]
    lap_telem_list = concat_laps_telemetry(d_laps)
    if not lap_telem_list:
        print(f"No usable telemetry for driver {d}, skipping.")
        continue
    driver_lap_map[d] = lap_telem_list
    all_lap_telem.extend(lap_telem_list)

In [67]:
def build_master_timeline(lap_telem_list: List[Tuple[int, pd.DataFrame, pd.Series]], freq_ms:int) -> np.ndarray:
    all_starts = []
    all_ends = []

    for drv in session.drivers:
        car = session.laps.pick_driver(drv)
        if len(car) == 0:
            continue

        try:
            df = car.get_car_data()
            t = df['Time'].dt.total_seconds()
            all_starts.append(t.min())
            all_ends.append(t.max())
        except Exception:
            pass

    if not all_starts:
        raise ValueError("Could not determine global telemetry time range")

    global_start = min(all_starts)
    global_end = max(all_ends)

    print(f"Global telemetry range: {global_start:.1f}s → {global_end:.1f}s")
    print(f"Total duration: {(global_end - global_start)/60:.2f} minutes")

    step = freq_ms / 1000.0
    return np.arange(global_start, global_end + 1e-9, step, dtype=float)

def resample_telemetry_to_timeline(telem: pd.DataFrame, tsec: pd.Series, timeline: np.ndarray) -> pd.DataFrame:
    df = telem.copy().reset_index(drop=True)

    df.index = pd.Index(tsec.astype(float), name='time_s')

    # Remove duplicate timestamps properly
    df = df.sort_index().groupby(level=0).last()

    # Split into categories
    numeric_cols = [c for c in INTERPOLATE_COLS if c in df.columns]
    categorical_cols = [c for c in CATEGORICAL_COLS if c in df.columns]

    # Create full timeline index
    timeline_index = pd.Index(timeline, name="time_s")
    full_index = df.index.union(timeline_index)

    # Reindex
    df_num = df[numeric_cols].reindex(full_index)
    df_cat = df[categorical_cols].reindex(full_index)

    # Interpolate numeric
    df_num = df_num.interpolate(method="index", limit_direction="both")

    # Fill categorical
    df_cat = df_cat.ffill().bfill()

    # Recombine
    df_resampled = pd.concat([df_num, df_cat], axis=1)

    # Extract only timeline values
    result = df_resampled.loc[timeline_index]
    result.index = result.index.astype(float)

    return result


In [68]:
timeline = build_master_timeline(all_lap_telem, freq_ms=200)

Global telemetry range: 0.1s → 8656.0s
Total duration: 144.26 minutes


/Users/br3nd4nt/F1Vision/DataGenerator/.venv/lib/python3.12/site-packages/fastf1/core.py:3183: FutureWarning: pick_driver is deprecated and will be removed in a future release. Use pick_drivers instead.
  warnings.warn(("pick_driver is deprecated and will be removed"


In [ ]:
for drv, lap_telem_list in driver_lap_map.items():
    print(f"Processing driver {drv} with {len(lap_telem_list)} laps")

    concat_rows = []
    concat_indicies = []
    concat_lap_ids = []
    for lap_idx, telem, tsec in lap_telem_list:
        concat_rows.append(telem.reset_index(drop=True))
        concat_indicies.append(tsec.reset_index(drop=True))
        concat_lap_ids.append(np.full(len(telem), lap_idx, dtype=int))
    
    if not concat_rows:
        print(f"No telemetry data for driver {drv}, skipping.")
        continue

    driver_telem = pd.concat(concat_rows, ignore_index=True)
    driver_tsec = pd.concat([pd.Series(idx) for idx in concat_indicies], ignore_index=True)
    driver_lapidx_col = np.concatenate(concat_lap_ids)

    print(f"Driver {drv} telemetry shape: {driver_telem.shape}, time shape: {driver_tsec.shape}")

    driver_telem.index = pd.Index(driver_tsec.astype(float), name='time_s')
    driver_telem['lap_idx'] = driver_lapidx_col

    driver_telem.sort_index().groupby(level=0).last()

    resampled = resample_telemetry_to_timeline(driver_telem, pd.Series(driver_telem.index), timeline)

    if 'lap_idx' in resampled.columns:
        lap_idx_values = resampled['lap_idx'].astype(int).fillna(-1).values
        resampled = resampled.drop(columns=['lap_idx'])
    else:
        lap_idx_values = np.full(len(resampled), -1, dtype=int)

    # fix driverAhead empty values
    # causes problems with first place driver
    # resampled["DriverAhead"] = resampled["DriverAhead"].replace(r"^\s*$", pd.NA, regex=True)
    # resampled["DriverAhead"] = resampled["DriverAhead"].ffill()

    print(f"Driver {drv} resampled telemetry shape: {resampled.shape}")
    display(resampled)


/var/folders/mp/3dq4rtz542d2jlnpvwjvkv7h0000gn/T/ipykernel_66251/574707375.py:51: FutureWarning: Telemetry.interpolate with object dtype is deprecated and will raise in a future version. Call obj.infer_objects(copy=False) before interpolating instead.
  df_num = df_num.interpolate(method="index", limit_direction="both")


Processing driver LEC with 78 laps
Driver LEC telemetry shape: (65794, 18), time shape: (65794,)
Driver LEC resampled telemetry shape: (43280, 14)


,Speed,RPM,Throttle,Brake,Distance,RelativeDistance,X,Y,Z,nGear,DRS,Source,Status,DriverAhead
time_s,,,,,,,,,,,,,,
0.087,0.000000,10499.368749,38.000000,NaN,0.001252,3.811788e-07,-7651.000291,-6789.990523,502.000000,1.0,1.0,interpolation,OnTrack,<NA>
0.287,0.000000,10499.368749,38.000000,NaN,0.001252,3.811788e-07,-7651.000291,-6789.990523,502.000000,1.0,1.0,interpolation,OnTrack,<NA>
0.487,0.000000,10499.368749,38.000000,NaN,0.001252,3.811788e-07,-7651.000291,-6789.990523,502.000000,1.0,1.0,interpolation,OnTrack,<NA>
0.687,0.000000,10499.368749,38.000000,NaN,0.001252,3.811788e-07,-7651.000291,-6789.990523,502.000000,1.0,1.0,interpolation,OnTrack,<NA>
0.887,0.000000,10499.368749,38.000000,NaN,0.001252,3.811788e-07,-7651.000291,-6789.990523,502.000000,1.0,1.0,interpolation,OnTrack,<NA>
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
8655.087,104.040752,7951.482759,30.896552,NaN,204.381111,6.204381e-02,-7254.060595,-4639.971922,539.092221,3.0,0.0,car,OnTrack,44
8655.287,104.667712,8020.448276,44.689655,NaN,210.214444,6.381463e-02,-7220.031917,-4628.985432,540.273497,3.0,0.0,car,OnTrack,44
8655.487,107.937503,8282.012709,59.050007,NaN,216.168192,6.562200e-02,-7176.114896,-4618.964884,542.330299,3.0,0.0,car,OnTrack,44


In [70]:
resampled.to_csv("LEC_resampled.csv", index=True)